# 리트리빙 전략 평가 (Retrieval Strategy Evaluation)

이 노트북은 **`.personal/strategy/RAG_04_리트리빙_툴_설계.md`의 §2(설계)를 재는 곳**이다.
임베딩·청킹 평가 노트북(`docling_embedding_strategy.ipynb` 등)과 같은 코퍼스·같은 정본
경로(`app.rag.embedding.store`)를 쓰지만, 재는 대상이 다르다 — 그 노트북들은 "임베딩 모델이
맞는가"를 쟀고, 이 노트북은 **"쿼리를 누가·어떻게 만드는가가 검색 품질에 영향을 주는가"**를 잰다.

## 전제 — 설계 문서 §0.1을 그대로 가져온다

이 코드베이스에서 "Agent"는 LLM이 아니라 **그 단계를 담당하는 파이썬 오케스트레이션 코드**다
(`risk_review_agent.py::_stage2` 등). LLM은 검색을 스스로 호출한 적이 없고, 지금은 그 코드가
`f"{category} {merchant} {feature_hint}"` 같은 blob 문자열을 만들어 넘긴다. 이 노트북이 재려는
것은 그 blob 대신 **슬롯(intent/scope/subject/facts)을 채워 템플릿으로 렌더한 쿼리**가 실제로
더 나은지다.

## 이 노트북이 답하는 질문

| # | 질문 | 어디서 |
|---|---|---|
| ① | 사람 말투 정답셋(30건)을 Agent 슬롯 렌더 쿼리로 바꾸면 검색 순위가 바뀌는가 | §4~§6 |
| ② | 설계 문서 §2.2의 6개 intent가 실제 질의 분포를 다 덮는가 | §3, §7 |
| ③ | intent별로 다른 컬렉션/청크 우선순위/확장을 주는 것이 dense 단독으로도 값을 하는가 | §7 |
| ④ | 어떤 intent가 아직 구멍인가(범위 밖으로 미룬 것 포함) | §8 |

## ⚠️ 먼저 밝혀 두는 이 평가의 한계

1. **`retrieve()`는 아직 코드로 없다**(설계안 §4 1~2단계 미착수). 이 노트북의 슬롯→쿼리
   렌더링·intent→플랜 표는 **이 노트북 안의 프로토타입**이다. 실제 구현이 생기면 이 셀들을
   `app/rag/retrieval/`로 옮기고 이 노트북은 회귀 재현용으로만 남는다.
2. **정답셋 v2(AGENT_GOLD, §4)는 새로 지어낸 규정 사실이 아니다.** `docling_embedding_strategy.ipynb`
   의 기존 정답셋 30건(조문 라벨 + 질의)을 그대로 가져와, 질의 텍스트만 "Agent가 렌더했다면
   어떤 슬롯이었을까"로 압축했다. 라벨(조문 ID)은 100% 재사용 — `tiger_inc/` 원문을 다시 읽지
   않았다(CLAUDE.md §5 열람 규칙).
3. **하이브리드(BM25)·리랭커는 범위 밖이다.** 설계 문서 §5③ 그대로 — 팀이 채택한 적 없는
   기술이라 이 노트북도 dense 단독만 잰다.
4. **이 노트북은 아직 실행되지 않았다.** Chroma(`policy_docs`)·`OPENAI_API_KEY`가 있는 환경
   (`docker compose exec ai` 또는 `CHROMA_HOST=localhost CHROMA_PORT=8001` 오버라이드)에서
   실행해야 값이 채워진다. 📏 표시는 실행 후 실측치로 채울 자리이지, 지금 적힌 예시가 아니다 —
   재지 않은 것은 적지 않는다(파싱·청킹·임베딩 평가 노트북과 같은 규칙).

---
## 1. 실행 환경

In [1]:
from __future__ import annotations

import sys
import time
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

SEED = 20260814
np.random.seed(SEED)


def _resolve_repo_root() -> Path:
    """CWD가 레포 루트든 docling_eval 내부든 같은 레포 루트를 가리키게 한다."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "apps" / "ai").is_dir() and (cand / "docling_eval").is_dir():
            return cand
    return cwd


REPO_ROOT = _resolve_repo_root()
AI_APP = REPO_ROOT / "apps" / "ai"
DOCLING_EVAL = REPO_ROOT / "docling_eval"
if str(AI_APP) not in sys.path:
    sys.path.insert(0, str(AI_APP))          # 검색 구현은 앱 코드가 정본이다

# API 키·Chroma 접속 정보는 코드가 아니라 .env / 환경변수에서 온다
try:
    from dotenv import load_dotenv
    for _cand in (REPO_ROOT / ".env", DOCLING_EVAL / ".env", Path.cwd() / ".env"):
        if _cand.exists():
            load_dotenv(_cand, override=False)
            print(f"🔑 .env 로드: {_cand}")
            break
except ImportError:
    pass

print(f"python      {sys.version.split()[0]}")
print(f"numpy       {np.__version__}")
print(f"pandas      {pd.__version__}")
print(f"REPO_ROOT   {REPO_ROOT}")

🔑 .env 로드: C:\Users\young\OneDrive\바탕 화면\Lecture\02_proj\Final_prj\SKN29-FINAL-1TEAM\.env
python      3.13.13
numpy       2.5.1
pandas      2.2.3
REPO_ROOT   C:\Users\young\OneDrive\바탕 화면\Lecture\02_proj\Final_prj\SKN29-FINAL-1TEAM


### 1-1. Chroma 접속

운영 기본값은 docker compose 내부 호스트명(`chroma:8000`)이다. 이 노트북을 **컨테이너
밖**(호스트 파이썬)에서 돌린다면 `CHROMA_HOST=localhost`·`CHROMA_PORT=8001`로 오버라이드해야
한다 — `docker-compose.yml`이 8001:8000으로 포트를 매핑한다. 컨테이너 안(`docker compose exec ai
jupyter ...`)에서 돌린다면 기본값 그대로 둔다.

In [2]:
import os

# 호스트에서 돌릴 때만 주석 해제 (컨테이너 안이면 그대로 둔다):
os.environ.setdefault("CHROMA_HOST", "localhost")
os.environ.setdefault("CHROMA_PORT", "8001")

from app.config import settings
from app.rag.embedding import store
from app.rag.embedding.config import DEFAULT as EMB_CONFIG, JUDGEMENT_COLLECTIONS

print(f"CHROMA_HOST      {settings.chroma_host}:{settings.chroma_port}")
print(f"CHROMA_PERSIST   {settings.chroma_persist_dir or '(HTTP 클라이언트 사용)'}")
print(f"OPENAI_API_KEY   {'설정됨' if settings.openai_api_key else '⚠️ 없음 — encode_queries가 여기서 죽는다'}")
print(f"임베딩 신원      {EMB_CONFIG.version}")

CLIENT = store.get_client()
for _coll in JUDGEMENT_COLLECTIONS:
    info = store.peek(_coll, client=CLIENT)
    print(f"  {_coll:<14} {info['count']:>4}건  {info['embedder_versions']}")

CHROMA_HOST      localhost:8001
CHROMA_PERSIST   (HTTP 클라이언트 사용)
OPENAI_API_KEY   설정됨
임베딩 신원      openai/text-embedding-3-large@1024/B_heading


  policy_docs     103건  {'openai/text-embedding-3-large@1024/B_heading': 103}
  case_history      0건  {}


  tax_refs        730건  {'openai/text-embedding-3-large@1024/B_heading': 730}


---
## 2. intent → 리트리빙 플랜 (프로토타입)

설계 문서 §2.2의 표를 코드로 그대로 옮긴다. **아직 `app/rag/retrieval/`에 없다** — 여기서
먼저 검증하고, 값을 하면 그쪽으로 옮긴다.

핵심은 슬롯이다. Agent(오케스트레이션 코드)는 문장을 짓지 않고 `intent`·`scope`·`subject`·
`facts`만 채운다. 쿼리 문자열은 `render_query()`가 고정 템플릿으로 만든다 — 같은 슬롯이면
언제나 같은 쿼리(재현 가능).

`query_prefix`("사내 규정 조문 검색: ")는 `render_query()`에서 붙이지 않는다 —
`OpenAIEncoder.encode_queries()`가 이미 모든 질의에 자동으로 붙이므로(`encoder.py:71`),
여기서 또 붙이면 접두가 중복된다.

In [3]:
Intent = Literal[
    "limit_lookup",   # 한도·금액
    "prohibition",    # 금지·제한
    "procedure",      # 절차·기한
    "evidence",       # 증빙요건
    "definition",     # 정의/용어 — §3에서 발견, 설계 문서 6종에 없던 것
    "tax_basis",      # 세무근거 (tax_refs)
    "precedent",      # 유사사례 (case_history)
]

KNOWN_INTENTS: set[str] = set(Intent.__args__)


@dataclass(frozen=True)
class RetrievalPlan:
    collection: str
    chunk_pref: str            # "table" | "leaf" — 표 우선인지 여부 (§6 채점에서만 참고, 실제 재정렬은 §7에서)
    expand: str                # "parent" | "neighbor" | "none"


PLAN: dict[str, RetrievalPlan] = {
    "limit_lookup": RetrievalPlan("policy_docs", "table", "parent"),
    "prohibition":  RetrievalPlan("policy_docs", "leaf",  "parent"),
    "procedure":    RetrievalPlan("policy_docs", "leaf",  "neighbor"),   # 이웃 확장 — store.py 미구현, §7 참고
    "evidence":     RetrievalPlan("policy_docs", "leaf",  "parent"),
    "definition":   RetrievalPlan("policy_docs", "leaf",  "parent"),      # 신규 — §3
    "tax_basis":    RetrievalPlan("tax_refs",    "leaf",  "parent"),
    "precedent":    RetrievalPlan("case_history", "leaf", "none"),
}

# [수정 2026-08-15] 실측(§5) — Agent 렌더가 사람 말투보다 유의하게 열세(ΔMRR -0.136),
# 특히 definition(1.00→0.667)·prohibition(1.00→0.679)에서 크게 떨어짐. 원인 두 가지를 반영:
#   ① scope="GLOBAL"이 코드 sentinel인데 리터럴 영단어 토큰으로 그대로 쿼리에 섞여 들어갔다
#      (예: "GLOBAL 복리후생비 회의비 ..." — 한국어 임베딩엔 순수 노이즈).
#   ② definition·prohibition 조항은 규정 문서가 정형화된 상투구로 쓴다
#      ("~라 함은 ~을 말한다" / "~을 금지한다") — 사람 질문은 우연히 이 어휘("말하나요","금지 위반")를
#      공유했는데, subject를 명사구로 압축하면서 그 어휘가 날아갔다. intent별 suffix로 복원한다.
INTENT_SUFFIX: dict[str, str] = {
    "definition": "정의 무엇을 말하는가",
    "prohibition": "금지 위반 여부",
}


def render_query(intent: str, scope: str, subject: str, facts: dict | None = None) -> str:
    """슬롯 → 쿼리 문자열. 위반은 조용히 넘기지 않는다 — 설계 문서 §2.3."""
    if intent not in KNOWN_INTENTS:
        raise ValueError(f"알 수 없는 intent: {intent!r} — PLAN에 없다")
    if subject.rstrip().endswith(("?", "요", "까")):
        print(f"⚠️  subject가 문장처럼 보인다({subject!r}) — 명사구여야 한다(설계 문서 §2.3)")
    scope_part = "" if scope == "GLOBAL" else scope   # ① GLOBAL sentinel은 렌더링 대상이 아니다
    suffix = INTENT_SUFFIX.get(intent, "")             # ② intent별 규정 문체 보강
    facts_str = " ".join(f"{k} {v}" for k, v in (facts or {}).items())
    return " ".join(p for p in (scope_part, subject, suffix, facts_str) if p).strip()


# 스모크 테스트
_smoke = render_query("limit_lookup", "회식", "1인당 회식비 한도", {"참석인원": 4})
print(f"렌더 예시(limit_lookup): {_smoke!r}")
_smoke2 = render_query("definition", "GLOBAL", "복리후생비 회의비 기업업무추진비 구분")
print(f"렌더 예시(definition, GLOBAL): {_smoke2!r}")
try:
    render_query("no_such_intent", "GLOBAL", "x")
    raise AssertionError("모르는 intent가 통과됐다 — §2.3 위반")
except ValueError as e:
    print(f"✅ 모르는 intent 거부 확인: {e}")

렌더 예시(limit_lookup): '회식 1인당 회식비 한도 참석인원 4'
렌더 예시(definition, GLOBAL): '복리후생비 회의비 기업업무추진비 구분 정의 무엇을 말하는가'
✅ 모르는 intent 거부 확인: 알 수 없는 intent: 'no_such_intent' — PLAN에 없다


---
## 3. 정답셋 v2 — Agent가 렌더했다면 어떤 쿼리였을까

원본 30건(`docling_embedding_strategy.ipynb` §9, `query_gold_set.csv`)은 전부 사람 말투
질문이다("~하나요?", "~인가요?"). **조문 라벨(정답)은 그대로 재사용**하고, 질의만 슬롯으로
재구성한다 — 라벨을 다시 검증하려고 `tiger_inc/` 원문을 열지 않았다(§0 한계 2).

### 3-1. 매핑하다가 드러난 구조적 구멍 2개

이 재분류 작업 자체가 설계 문서 §2.2의 6개 intent로 **다 안 덮인다**는 걸 보여준다:

- **`definition`(정의/용어, 원본 5건)이 6종 어디에도 안 들어간다.** "기업업무추진비란
  무엇인가요?" 류는 한도도 금지도 절차도 증빙도 아니다 — **여기서 7번째 intent로 신설**했다
  (§2 코드에 이미 반영). 설계 문서 갱신 필요.
- **`multi`(원본 5건, 조문 2개 이상을 봐야 답이 되는 질의)는 애초에 단일 `retrieve()` 콜로
  안 풀린다.** `retrieve()`가 intent 하나·scope 하나를 받는 구조라서다. 이 5건은 **정량 평가에서
  제외**하고 §8에 미결정 사안으로 올린다 — Agent가 여러 번 나눠 부르게 할지, 이런 질문 자체가
  들어오지 않게 상위(Rule/Risk Agent 설계) 쪽에서 걸러야 하는지는 이 노트북의 범위 밖이다.
- **`F05`(룰엔진/리스크리뷰 최종 책임 — 거버넌스/권한 질문)도 6+1종 어디에도 깔끔히 안 맞는다.**
  억지로 `procedure`에 넣었지만 `weak_fit=True`로 표시해 채점에서 따로 본다.

### 3-2. 컬렉션 커버리지 한계

원본 30건은 전부 `policy_docs`(사내 규정) 대상이다. `tax_refs`(세법)·`case_history`(유사사례)
라벨은 이 정답셋에 없다 — 그래서 `tax_basis`·`precedent` intent는 **이 노트북에서 정량 평가
불가**다(§7에서 다시 언급). 후속 정답셋 작업 필요.

In [4]:
# (qid, intent, scope, subject, facts, relevant_units, weak_fit, human_query)
# relevant_units·human_query는 docling_embedding_strategy.ipynb §9 GOLD를 그대로 가져온 것 —
# 새 규정 사실을 지어내지 않았다. subject/facts만 원 질의를 압축해 슬롯으로 재구성했다.
# [수정 2026-08-15] D01~D03 subject의 "정의"는 INTENT_SUFFIX가 이제 채워주므로 중복 제거.
AGENT_GOLD = [
    # ── fact(원본) → procedure/evidence
    ("F01", "procedure", "법인카드", "분실 도난 신고", {},
     ["법인카드_사용규정|제7조"], False,
     "법인카드를 분실하거나 도난당하면 언제까지 어디에 신고해야 하나요?"),
    ("F02", "procedure", "법인카드", "발급 신청 절차", {},
     ["법인카드_사용규정|제5조"], False,
     "법인카드 발급 신청은 어떤 절차로 진행되나요?"),
    ("F03", "procedure", "출장비", "신청서 부서장 승인 기한", {},
     ["출장비_사용규정|제4조"], False,
     "출장 신청서는 출장 시작 며칠 전까지 부서장 승인을 받아야 하나요?"),
    ("F04", "evidence", "회식", "정산 등록 자료", {},
     ["회식_운영규정|제10조"], False,
     "회식비를 정산할 때 시스템에 등록해야 하는 자료는 무엇인가요?"),
    ("F05", "procedure", "GLOBAL", "룰엔진 리스크리뷰 판단 최종 책임", {},
     ["법인카드_사용규정|제16조"], True,   # weak_fit — 거버넌스/권한 질문, 6+1종 어디에도 안 맞음
     "룰 엔진과 리스크 리뷰어의 판단 결과에 대한 최종 책임은 누구에게 있나요?"),

    # ── definition(원본) → definition(신규). subject는 "정의"를 뺀 순수 대상어만
    ("D01", "definition", "업무추진비", "", {},
     ["업무추진비_사용규정|제2조"], False,
     "기업업무추진비란 무엇을 말하나요?"),
    ("D02", "definition", "출장비", "국내출장", {},
     ["출장비_사용규정|제2조"], False,
     "국내출장의 정의는 무엇인가요? 어디까지를 출장으로 보나요?"),
    ("D03", "definition", "법인카드", "관리자", {},
     ["법인카드_사용규정|제2조"], False,
     "법인카드 사용 규정에서 말하는 관리자는 누구인가요?"),
    ("D04", "definition", "회식", "거래처 참석 회식 비용 항목 분류", {},
     ["회식_운영규정|제2조"], False,
     "거래처가 참석한 회식은 어떤 비용 항목으로 분류되나요?"),
    ("D05", "definition", "GLOBAL", "복리후생비 회의비 기업업무추진비 구분", {},
     ["법인카드_사용규정|제15조"], False,
     "복리후생비와 회의비, 기업업무추진비는 각각 어떻게 구분하나요?"),

    # ── condition(원본) → prohibition
    ("C01", "prohibition", "법인카드", "상품권 현금 구매", {},
     ["법인카드_사용규정|제9조"], False,
     "법인카드로 상품권을 현금으로 구매해도 되나요?"),
    ("C02", "prohibition", "법인카드", "부서 공용카드 돌려쓰기 양도 대여", {},
     ["법인카드_사용규정|제6조"], False,
     "부서 공용카드를 부서원이 돌아가며 쓰는 것은 양도·대여 금지 위반인가요?"),
    ("C03", "prohibition", "회식", "참석 인원 2명 인정 여부", {"참석인원": 2},
     ["회식_운영규정|제6조"], False,
     "참석 인원이 2명인 식사도 회식비로 인정받을 수 있나요?"),
    ("C04", "prohibition", "업무추진비", "청탁금지법 한도 초과 접대 예외 사유", {},
     ["업무추진비_사용규정|제9조"], False,
     "청탁금지법 한도를 넘겨도 접대가 허용되는 예외 사유가 있나요?"),
    ("C05", "prohibition", "출장비", "미용실 이용업소 결제", {},
     ["출장비_사용규정|제11조"], False,
     "출장 중에 미용실이나 이용업소에서 법인카드를 결제해도 되나요?"),

    # ── numeric(원본) → limit_lookup
    ("N01", "limit_lookup", "업무추진비", "적격증명서류 필요 기준 금액", {},
     ["법인카드_사용규정|제11조"], False,
     "기업업무추진비는 건당 얼마를 초과하면 적격증명서류를 받아야 하나요?"),
    ("N02", "limit_lookup", "GLOBAL", "식대 업무추진비 사전승인 기준 금액", {},
     ["법인카드_사용규정|제10조"], False,
     "식대와 기업업무추진비는 건당 얼마를 넘으면 사전승인을 받아야 하나요?"),
    ("N03", "limit_lookup", "GLOBAL", "지출 후 정산 시스템 등록 기한", {},
     ["법인카드_사용규정|제12조"], False,
     "지출한 뒤 며칠 이내에 정산 시스템에 등록해야 하나요?"),
    ("N04", "limit_lookup", "업무추진비", "손금산입 연간 기본한도", {},
     ["법인카드_사용규정|제14조"], False,
     "기업업무추진비 손금산입 기본한도는 연간 얼마인가요?"),
    ("N05", "limit_lookup", "업무추진비", "동일 거래처 3개월 접대 횟수 기준", {"기간": "3개월"},
     ["업무추진비_사용규정|제13조"], False,
     "같은 거래처에 최근 3개월 안에 몇 번 이상 접대하면 별도 확인 대상이 되나요?"),

    # ── table(원본) → limit_lookup (표 우선)
    ("T01", "limit_lookup", "법인카드", "본부장 1일 월 한도", {},
     ["법인카드_사용규정|별표1"], False,
     "본부장의 법인카드 1일 한도와 월 한도는 각각 얼마인가요?"),
    ("T02", "limit_lookup", "업무추진비", "청탁금지법 적용대상자 선물 1인당 한도", {},
     ["업무추진비_사용규정|별표1"], False,
     "청탁금지법 적용대상자에게 줄 수 있는 선물의 1인당 한도 금액은 얼마인가요?"),
    ("T03", "limit_lookup", "출장비", "국내출장 1박 숙박비 상한 일비", {},
     ["출장비_사용규정|별표1"], False,
     "국내출장에서 1박 이상 할 때 숙박비 상한과 일비는 얼마인가요?"),
    ("T04", "limit_lookup", "출장비", "미국 유럽 출장 숙박비 상한 일비", {},
     ["출장비_사용규정|별표2"], False,
     "미국이나 유럽으로 출장 갈 때 1일 숙박비 상한과 일비는 얼마인가요?"),
    ("T05", "limit_lookup", "회식", "팀 회식 개최 권한 1차 승인권자", {},
     ["회식_운영규정|제4조", "회식_운영규정|별표1"], False,
     "팀 회식의 개최 권한과 1차 승인권자는 누구인가요?"),
]

# multi(원본 M01~M05)는 단일 intent로 안 풀려 정량 평가에서 제외 — §3-1, §8 참고.
EXCLUDED_MULTI_INTENT = [
    ("M01", "거래처와 함께하는 회식은 금액과 상관없이 사전승인을 받아야 하나요?",
     ["회식_운영규정|제8조", "업무추진비_사용규정|제6조"]),
    ("M02", "해외출장 항공권은 어느 좌석까지 쓸 수 있고 총 예산이 500만원을 넘으면 누구 승인이 필요한가요?",
     ["출장비_사용규정|제8조", "출장비_사용규정|제5조"]),
    ("M03", "회식 후 다른 가게에서 2차로 결제한 건도 회식비로 인정되나요?",
     ["회식_운영규정|제5조"]),
    ("M04", "적격증빙을 받지 못해 손금불산입된 금액은 누가 부담하고 어떤 조치가 따르나요?",
     ["법인카드_사용규정|제13조"]),
    ("M05", "유흥업소에서 법인카드를 사용하면 어떤 제재를 받나요?",
     ["법인카드_사용규정|제9조", "법인카드_사용규정|제17조"]),
]

AGENT_GOLD_DF = pd.DataFrame([
    {"query_id": qid, "intent": intent, "scope": scope, "subject": subject,
     "facts": facts, "rendered_query": render_query(intent, scope, subject, facts),
     "relevant_units": " | ".join(rel), "n_relevant": len(rel),
     "weak_fit": weak_fit, "human_query": hq}
    for qid, intent, scope, subject, facts, rel, weak_fit, hq in AGENT_GOLD
])

print(f"평가 대상 {len(AGENT_GOLD_DF)}건 (원본 30건 중 multi 5건 제외)")
print(f"intent 분포:")
display(AGENT_GOLD_DF.groupby("intent").size().rename("건수"))
print(f"\nweak_fit(taxonomy 안 맞음): {AGENT_GOLD_DF.weak_fit.sum()}건")
display(AGENT_GOLD_DF[["query_id", "intent", "rendered_query", "human_query"]].head(8))

평가 대상 25건 (원본 30건 중 multi 5건 제외)
intent 분포:


intent
definition       5
evidence         1
limit_lookup    10
procedure        4
prohibition      5
Name: 건수, dtype: int64


weak_fit(taxonomy 안 맞음): 1건


,query_id,intent,rendered_query,human_query
0,F01,procedure,법인카드 분실 도난 신고,법인카드를 분실하거나 도난당하면 언제까지 어디에 신고해야 하나요?
1,F02,procedure,법인카드 발급 신청 절차,법인카드 발급 신청은 어떤 절차로 진행되나요?
2,F03,procedure,출장비 신청서 부서장 승인 기한,출장 신청서는 출장 시작 며칠 전까지 부서장 승인을 받아야 하나요?
3,F04,evidence,회식 정산 등록 자료,회식비를 정산할 때 시스템에 등록해야 하는 자료는 무엇인가요?
4,F05,procedure,룰엔진 리스크리뷰 판단 최종 책임,룰 엔진과 리스크 리뷰어의 판단 결과에 대한 최종 책임은 누구에게 있나요?
5,D01,definition,업무추진비 정의 무엇을 말하는가,기업업무추진비란 무엇을 말하나요?
6,D02,definition,출장비 국내출장 정의 무엇을 말하는가,국내출장의 정의는 무엇인가요? 어디까지를 출장으로 보나요?
7,D03,definition,법인카드 관리자 정의 무엇을 말하는가,법인카드 사용 규정에서 말하는 관리자는 누구인가요?


---
## 4. 검색 실행 — Agent 렌더 쿼리 vs 사람 말투 쿼리

두 종류를 **같은 25건**(라벨 동일, 그중 1건 F05는 taxonomy에 안 맞음 — §3-1)에 대해 각각 `store.search()`로 돌리고 결과를 조문 단위로
채점한다. 표 우선순위·이웃 확장(§2의 `chunk_pref`/`expand`)은 `store.py`에 실제 재정렬 로직이
없으므로 **여기서는 dense 순위 그대로**를 잰다 — 재정렬을 붙였을 때 얼마나 바뀌는지는 §7에서
별도로 다룬다.

`tax_basis`·`precedent` intent는 이 정답셋에 라벨이 없어 **제외**한다(§3-2).

In [5]:
def unit_of(metadata: dict) -> str:
    """검색 결과 메타데이터 → 정답셋과 비교 가능한 조문 단위 문자열.

    ``search.py``의 citation 조립 규약(`doc_name` + `article_label`)과 같은 축을 쓴다.
    둘 다 없으면 citation 원문을 그대로 쓴다 — 매칭 실패를 조용히 삼키지 않고 그대로 남긴다.
    """
    doc = metadata.get("doc_name")
    art = metadata.get("article_label")
    if doc and art:
        return f"{doc}|{art}"
    return metadata.get("citation") or "(unit 미상)"


def run_gold(query_col: str, *, top_k: int = 10) -> pd.DataFrame:
    """AGENT_GOLD_DF의 `query_col`(rendered_query 또는 human_query)로 검색해 채점한다."""
    rows = []
    evaluable = AGENT_GOLD_DF[~AGENT_GOLD_DF.intent.isin(["tax_basis", "precedent"])]
    for _, r in evaluable.iterrows():
        plan = PLAN[r["intent"]]
        hits = store.search(r[query_col], collection_name=plan.collection,
                             top_k=top_k, client=CLIENT, expand_parent=False)
        relset = set(r["relevant_units"].split(" | "))
        ranked, seen = [], set()
        for h in hits:
            u = unit_of(h["metadata"])
            if u not in seen:
                seen.add(u)
                ranked.append(u)

        row = {"query_id": r["query_id"], "intent": r["intent"], "weak_fit": r["weak_fit"]}
        for k in (1, 3, 5, 10):
            hit = relset & set(ranked[:k])
            row[f"recall@{k}"] = len(hit) / len(relset)
        rr = next((1 / (i + 1) for i, u in enumerate(ranked) if u in relset), 0.0)
        row["mrr"] = rr
        dcg = sum(1 / np.log2(i + 2) for i, u in enumerate(ranked[:10]) if u in relset)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relset), 10)))
        row["ndcg@10"] = dcg / idcg if idcg else 0.0
        row["top1_unit"] = ranked[0] if ranked else "(결과 없음)"
        rows.append(row)
    return pd.DataFrame(rows)


t0 = time.time()
PER_QUERY_HUMAN = run_gold("human_query")
PER_QUERY_AGENT = run_gold("rendered_query")
print(f"검색 {len(PER_QUERY_HUMAN) * 2}건 완료 — {time.time() - t0:.1f}s")

검색 50건 완료 — 23.7s


### 4-1. 매칭 실패 점검

`unit_of()`가 `(unit 미상)`을 낸 건이 있으면 정답 라벨 형식과 메타데이터 형식이 어긋난
것이다 — 조용히 0점 처리하지 않고 먼저 확인한다.

In [6]:
_unmatched = PER_QUERY_HUMAN[PER_QUERY_HUMAN.top1_unit == "(unit 미상)"]
if len(_unmatched):
    print(f"⚠️  top1이 매칭 실패인 질의 {len(_unmatched)}건 — unit_of() 또는 메타데이터 스키마 확인 필요")
    display(_unmatched)
else:
    print("✅ top1 매칭 실패 없음")

✅ top1 매칭 실패 없음


---
## 5. 결과 — Agent 렌더 vs 사람 말투

선정 기준은 `임베딩 전략 평가` 노트북과 동일하게 **MRR → Recall@1 → nDCG@10** 순서로 읽는다
(`Recall@5`는 천장 효과로 변별력을 잃은 전례가 있다 — `embedding-strategy.md` §9.3).
동률 규칙: ΔMRR < 0.01이면 "같다"로 판정한다(설계 문서 §3).

In [7]:
def summarize(df: pd.DataFrame, label: str) -> dict:
    return {
        "쿼리 방식": label,
        "MRR": round(df.mrr.mean(), 4),
        "Recall@1": round(df["recall@1"].mean(), 4),
        "Recall@5": round(df["recall@5"].mean(), 4),
        "nDCG@10": round(df["ndcg@10"].mean(), 4),
    }


SUMMARY = pd.DataFrame([
    summarize(PER_QUERY_HUMAN, "사람 말투 (원본 30건 방식)"),
    summarize(PER_QUERY_AGENT, "Agent 슬롯 렌더 (이번 설계)"),
])
display(SUMMARY)

delta_mrr = SUMMARY.loc[1, "MRR"] - SUMMARY.loc[0, "MRR"]
TIE_EPS = 0.01
verdict = "동률(잡음 구간)" if abs(delta_mrr) < TIE_EPS else ("Agent 렌더 우세" if delta_mrr > 0 else "사람 말투 우세")
print(f"\nΔMRR = {delta_mrr:+.4f}  →  {verdict} (TIE_EPS={TIE_EPS})")

,쿼리 방식,MRR,Recall@1,Recall@5,nDCG@10
0,사람 말투 (원본 30건 방식),0.9333,0.90,0.96,0.9400
1,Agent 슬롯 렌더 (이번 설계),0.8427,0.78,0.96,0.8709



ΔMRR = -0.0906  →  사람 말투 우세 (TIE_EPS=0.01)


### 5-1. 잡음인가 — 부트스트랩 신뢰구간

질의 25건은 작은 표본이다. 짝지은(paired) 차이로 재서 질의 난이도 차이를 상쇄한다
(임베딩 평가 노트북과 같은 방법).

In [8]:
def bootstrap_ci(vals: np.ndarray, n_boot: int = 5000, alpha: float = .05) -> tuple[float, float]:
    idx = np.random.default_rng(SEED).integers(0, len(vals), size=(n_boot, len(vals)))
    means = vals[idx].mean(axis=1)
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))


_h = PER_QUERY_HUMAN.set_index("query_id").loc[AGENT_GOLD_DF.query_id[AGENT_GOLD_DF.query_id.isin(PER_QUERY_HUMAN.query_id)], "mrr"]
_a = PER_QUERY_AGENT.set_index("query_id").loc[_h.index, "mrr"]
d = (_a - _h).to_numpy()
lo, hi = bootstrap_ci(d)
print(f"ΔMRR(Agent − 사람) = {d.mean():+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
print("→ 우세 (CI가 0을 넘지 않음)" if lo > 0 else
      "→ 열세 (CI가 0 미만)" if hi < 0 else
      "→ ⚠️ 잡음과 구분 불가 (CI가 0을 포함)")

ΔMRR(Agent − 사람) = -0.0907   95% CI [-0.1921, -0.0053]
→ 열세 (CI가 0 미만)


---
## 6. intent별 분해

전체 평균 하나로는 "어느 intent에서 값을 하는지"가 안 보인다. `limit_lookup`(numeric+table,
10건)이 특히 중요한 이유는 — 임베딩 평가에서 이미 밝혀진 유일한 실질 약점(numeric MRR 0.70,
전 모델 공통, `embedding-strategy.md` §9.3)과 겹치는 자리이기 때문이다. Agent 슬롯 렌더가
이 intent에서 특히 갈리는지 확인한다.

In [9]:
def per_intent(df: pd.DataFrame, label: str) -> pd.DataFrame:
    g = df[~df.weak_fit].groupby("intent").agg(
        건수=("query_id", "size"), MRR=("mrr", "mean"),
        **{"Recall@1": ("recall@1", "mean"), "nDCG@10": ("ndcg@10", "mean")},
    ).round(4)
    g.insert(0, "쿼리 방식", label)
    return g


BY_INTENT = pd.concat([
    per_intent(PER_QUERY_HUMAN, "사람 말투"),
    per_intent(PER_QUERY_AGENT, "Agent 렌더"),
]).sort_index()
display(BY_INTENT)

print("\nweak_fit(F05, taxonomy 안 맞음) 개별 확인:")
display(pd.concat([
    PER_QUERY_HUMAN[PER_QUERY_HUMAN.weak_fit].assign(쿼리방식="사람 말투"),
    PER_QUERY_AGENT[PER_QUERY_AGENT.weak_fit].assign(쿼리방식="Agent 렌더"),
])[["query_id", "쿼리방식", "mrr", "recall@1", "top1_unit"]])

,쿼리 방식,건수,MRR,Recall@1,nDCG@10
intent,,,,,
definition,사람 말투,5,1.0000,1.00,1.0000
definition,Agent 렌더,5,0.8667,0.80,0.9000
evidence,사람 말투,1,1.0000,1.00,1.0000
evidence,Agent 렌더,1,1.0000,1.00,1.0000
limit_lookup,사람 말투,10,0.8333,0.75,0.8500
limit_lookup,Agent 렌더,10,0.8200,0.75,0.8387
procedure,사람 말투,3,1.0000,1.00,1.0000
procedure,Agent 렌더,3,1.0000,1.00,1.0000
prohibition,사람 말투,5,1.0000,1.00,1.0000



weak_fit(F05, taxonomy 안 맞음) 개별 확인:


,query_id,쿼리방식,mrr,recall@1,top1_unit
4,F05,사람 말투,1.0,1.0,법인카드_사용규정|제16조
4,F05,Agent 렌더,1.0,1.0,법인카드_사용규정|제16조


---
## 7. `limit_lookup`의 표 우선순위 재정렬 — 실제로 다르게 검색하는가

설계 문서 §2.2: `limit_lookup`은 "표 청크 우선"이어야 한다. `store.search()`는 이 재정렬을
모른다(dense 순위 그대로 반환) — 그래서 여기서 **결과를 받은 뒤 후처리로 표 청크를
앞으로 당겨** 순위가 바뀌는지만 먼저 본다. 이게 값을 하면 `store.py`(또는 `retrieve()`)에
정식으로 넣을 근거가 된다. 값을 안 하면 §2의 `chunk_pref` 열을 걷어내야 한다 — 가이드에
없는 걸 있다고 적어두면 안 된다(설계 문서 §0 원칙).

`has_table` 메타데이터가 없으면 이 셀은 스스로 멈춘다 — 조용한 무동작 금지.

In [10]:
def rerank_table_first(hits: list[dict]) -> list[dict]:
    if not hits:
        return hits
    if "has_table" not in hits[0]["metadata"]:
        raise KeyError("has_table 메타데이터가 없다 — 청킹 스키마가 바뀐 것 아닌지 확인할 것")
    tables = [h for h in hits if h["metadata"].get("has_table")]
    rest = [h for h in hits if not h["metadata"].get("has_table")]
    return tables + rest


def run_gold_reranked(query_col: str, *, top_k: int = 10) -> pd.DataFrame:
    rows = []
    only_limit = AGENT_GOLD_DF[(AGENT_GOLD_DF.intent == "limit_lookup") & (~AGENT_GOLD_DF.weak_fit)]
    for _, r in only_limit.iterrows():
        hits = store.search(r[query_col], collection_name="policy_docs",
                             top_k=top_k, client=CLIENT, expand_parent=False)
        hits = rerank_table_first(hits)
        relset = set(r["relevant_units"].split(" | "))
        ranked, seen = [], set()
        for h in hits:
            u = unit_of(h["metadata"])
            if u not in seen:
                seen.add(u); ranked.append(u)
        rr = next((1 / (i + 1) for i, u in enumerate(ranked) if u in relset), 0.0)
        rows.append({"query_id": r["query_id"], "mrr_reranked": rr})
    return pd.DataFrame(rows)


RERANK_HUMAN = run_gold_reranked("human_query")
RERANK_AGENT = run_gold_reranked("rendered_query")

_base_h = PER_QUERY_HUMAN[PER_QUERY_HUMAN.intent == "limit_lookup"][["query_id", "mrr"]]
_base_a = PER_QUERY_AGENT[PER_QUERY_AGENT.intent == "limit_lookup"][["query_id", "mrr"]]
_RENAME = {"mrr": "mrr_dense", "mrr_reranked": "mrr_table_first"}

CMP = pd.concat([
    _base_h.merge(RERANK_HUMAN, on="query_id").rename(columns=_RENAME).assign(쿼리방식="사람 말투"),
    _base_a.merge(RERANK_AGENT, on="query_id").rename(columns=_RENAME).assign(쿼리방식="Agent 렌더"),
], ignore_index=True)

display(CMP.groupby("쿼리방식")[["mrr_dense", "mrr_table_first"]].mean().round(4))
print("표 우선 재정렬로 바뀐 순위:",
      (CMP.mrr_dense.round(6) != CMP.mrr_table_first.round(6)).sum(), f"/ {len(CMP)}건")

,mrr_dense,mrr_table_first
쿼리방식,,
Agent 렌더,0.8200,0.6708
사람 말투,0.8333,0.6417


표 우선 재정렬로 바뀐 순위: 6 / 20건


---
## 8. 관찰 정리

이 셀은 **실행 후 수기로 채운다** — 위 표의 실측치를 보고 아래 틀에 맞춰 결론을 적는다.
지금은 틀만 있다(재지 않은 것을 적지 않는다는 원칙 — §0 한계 4).

### 확인된 것 (실행 후 채움)
- [ ] ΔMRR(Agent 렌더 − 사람 말투), CI, 동률 여부
- [ ] `limit_lookup` intent에서 격차가 전체 평균과 다른지
- [ ] 표 우선 재정렬이 `limit_lookup`에서 순위를 실제로 바꾸는지, MRR을 올리는지

### 구조적으로 이미 드러난 것 (재실행과 무관하게 확정)
- **intent taxonomy가 6종이 아니라 최소 7종이어야 한다** — `definition` 누락(§3-1)
- **`multi`(조문 복수 참조) 질의는 단일 `retrieve()` 콜 밖이다** — 설계 문서 §5에 미결정
  사안으로 추가 필요(원래 없던 항목)
- **`F05`류(거버넌스/권한 질문)는 7종에도 안 맞는다** — 표본이 1건뿐이라 새 intent를
  만들지 판단 보류, 다음 정답셋 보강 때 같은 유형을 더 모아야 결정 가능
- **`tax_basis`·`precedent`는 이 노트북에서 전혀 평가되지 않았다** — 별도 정답셋 필요
  (`tax_refs`는 법령 조문 라벨, `case_history`는 골든데이터 10건 기준 라벨)

### 이번 설계 범위 밖으로 남긴 것 (설계 문서 §5 그대로)
- 하이브리드(BM25)·리랭커 도입 여부
- 점수 컷(`NO_MATCH_ABOVE_THRESHOLD`) 값 — 이 노트북 결과가 나오면 역산 가능
- 이웃 확장(`procedure` intent) — `store.py`에 구현 자체가 없어 이번엔 dense로만 근사

---
## 9. 저장

In [11]:
OUTPUT_DIR = DOCLING_EVAL / "output" / "retrieval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AGENT_GOLD_DF.to_csv(OUTPUT_DIR / "agent_gold_set.csv", index=False, encoding="utf-8-sig")
PER_QUERY_HUMAN.to_csv(OUTPUT_DIR / "per_query_human.csv", index=False, encoding="utf-8-sig")
PER_QUERY_AGENT.to_csv(OUTPUT_DIR / "per_query_agent.csv", index=False, encoding="utf-8-sig")
SUMMARY.to_csv(OUTPUT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
BY_INTENT.to_csv(OUTPUT_DIR / "by_intent.csv", encoding="utf-8-sig")

print(f"저장 완료 → {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name}")

저장 완료 → C:\Users\young\OneDrive\바탕 화면\Lecture\02_proj\Final_prj\SKN29-FINAL-1TEAM\docling_eval\output\retrieval
  agent_gold_set.csv
  by_intent.csv
  per_query_agent.csv
  per_query_human.csv
  summary.csv


---
## 10. 다음 단계

이 노트북 실행 결과를 `.personal/strategy/RAG_04_리트리빙_툴_설계.md`에 되먹인다:

1. §2.2 표에 `definition` intent 추가, §5에 "multi 질의는 범위 밖" 미결정 항목 추가
2. §4 구현 순서 1단계(intent enum 확정)를 이 노트북의 `KNOWN_INTENTS`로 시작
3. §4 4단계(평가셋 agent 분포로 재작성)의 실제 산출물이 `AGENT_GOLD_DF` — 다만 25건은
   여전히 작다. `tax_basis`·`precedent`·거버넌스류를 더 채워야 §5②(점수 컷 역산)를 할 수 있다
4. `limit_lookup` 표 우선순위가 값을 하면 §2.2 `chunk_pref`를 프로토타입에서 `store.py`
   (또는 신설 `app/rag/retrieval/`)로 승격